In [1]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

# Mount Google Drive to access the datasets, model files, and outputs.
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import torch
import numpy as np
import pandas as pd
import os

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')


# ============================================================
# DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )


# ============================================================
# PATHS
# ============================================================

AUGMENTED_DATASET_DIR = (
    "/content/drive/MyDrive/code_switch_project/data/augmented_dataset"
)

MODEL_OUTPUT_DIR = (
    "/content/drive/MyDrive/code_switch_project/models/xlm_roberta_code_switching"
)

RESULTS_OUTPUT_DIR = (
    "/content/drive/MyDrive/code_switch_project/results"
)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_OUTPUT_DIR, exist_ok=True)


# ============================================================
# FINE-TUNING PARAMETERS
# ============================================================

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 4
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
SEED = 42


# Set random seeds for reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda
GPU: Tesla T4
GPU Memory: 15.64 GB


In [3]:
# ============================================================
# LOAD DATASETS
# ============================================================

print("\nLoading datasets...")

train_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "train_augmented.csv")
)

val_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "val_augmented.csv")
)

test_clean_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "test_clean.csv")
)

test_augmented_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "test_augmented.csv")
)

print(f"Train: {len(train_df):,} rows")
print(f"Val: {len(val_df):,} rows")
print(f"Test (clean): {len(test_clean_df):,} rows")
print(f"Test (augmented): {len(test_augmented_df):,} rows")


# ============================================================
# LABEL MAPPING
# ============================================================

# Map the three language classes to numerical labels for training.
label2id = {
    'french': 0,
    'german': 1,
    'mixed': 2
}

# Reverse mapping used to convert model predictions back to class names.
id2label = {
    0: 'french',
    1: 'german',
    2: 'mixed'
}

num_labels = len(label2id)

print(f"\nLabel mapping: {label2id}")


# ============================================================
# LOAD XLM-ROBERTA MODEL AND TOKENIZER
# ============================================================

print(f"\nLoading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

print(f"Model loaded and moved to {device}")
print(
    f"Total parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


Loading datasets...
Train: 212,022 rows
Val: 43,224 rows
Test (clean): 39,740 rows
Test (augmented): 39,740 rows

Label mapping: {'french': 0, 'german': 1, 'mixed': 2}

Loading xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cuda
Total parameters: 278,045,955


In [4]:
# ============================================================
# CUSTOM DATASET CLASS
# ============================================================

class CodeSwitchingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = [label2id[label] for label in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenize and prepare the input for XLM-RoBERTa.
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# ============================================================
# CREATE TRAINING, VALIDATION, AND TEST DATASETS
# ============================================================

print("Creating datasets...")

train_dataset = CodeSwitchingDataset(
    train_df['text'].values,
    train_df['language'].values,
    tokenizer,
    MAX_LENGTH
)

val_dataset = CodeSwitchingDataset(
    val_df['text'].values,
    val_df['language'].values,
    tokenizer,
    MAX_LENGTH
)

test_clean_dataset = CodeSwitchingDataset(
    test_clean_df['text'].values,
    test_clean_df['language'].values,
    tokenizer,
    MAX_LENGTH
)

test_augmented_dataset = CodeSwitchingDataset(
    test_augmented_df['text'].values,
    test_augmented_df['language'].values,
    tokenizer,
    MAX_LENGTH
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")
print(f"Test (clean) dataset: {len(test_clean_dataset)} samples")
print(f"Test (augmented) dataset: {len(test_augmented_dataset)} samples")

Creating datasets...
Train dataset: 212022 samples
Val dataset: 43224 samples
Test (clean) dataset: 39740 samples
Test (augmented) dataset: 39740 samples


In [5]:

# ============================================================
# CREATE DATA LOADERS
# ============================================================

print("\nCreating data loaders...")

# Shuffle training data so batches contain varied examples
# during each training epoch.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

# Keep validation and test data in a fixed order because
# shuffling is not required during evaluation.
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_clean_loader = DataLoader(
    test_clean_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_augmented_loader = DataLoader(
    test_augmented_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")
print(f"Test (clean) loader: {len(test_clean_loader)} batches")
print(f"Test (augmented) loader: {len(test_augmented_loader)} batches")


# ============================================================
# VERIFY TRAINING BATCH
# ============================================================

print("\nVerifying one training batch...")

sample_batch = next(iter(train_loader))

print(f"  input_ids shape: {sample_batch['input_ids'].shape}")
print(f"  attention_mask shape: {sample_batch['attention_mask'].shape}")
print(f"  labels shape: {sample_batch['labels'].shape}")
print(
    f"  labels unique values: "
    f"{torch.unique(sample_batch['labels']).tolist()}"
)


Creating data loaders...
Train loader: 6626 batches
Val loader: 1351 batches
Test (clean) loader: 1242 batches
Test (augmented) loader: 1242 batches

Verifying one training batch...
  input_ids shape: torch.Size([32, 128])
  attention_mask shape: torch.Size([32, 128])
  labels shape: torch.Size([32])
  labels unique values: [0, 1, 2]


In [6]:
# ============================================================
# OPTIMIZER AND LEARNING-RATE SCHEDULER
# ============================================================

print("Setting up optimizer and scheduler...")

# AdamW updates the model parameters during fine-tuning.
# Weight decay helps reduce overfitting.
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Calculate the total number of parameter-update steps
# across all training epochs.
total_steps = len(train_loader) * NUM_EPOCHS

# Use a linear learning-rate schedule with a warmup phase.
# The learning rate gradually increases during warmup and
# then decreases linearly over the remaining training steps.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {WARMUP_STEPS}")

Setting up optimizer and scheduler...
Total training steps: 26,504
Warmup steps: 500


In [7]:
# ============================================================
# TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, device):
    """
    Train the model for one complete epoch.

    Returns:
        Average training loss and training accuracy.
    """

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(loader):

        # Move the current batch to the selected device (GPU/CPU).
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # --------------------------------------------------------
        # Forward pass
        # --------------------------------------------------------
        # The model processes the input and calculates the
        # classification loss using the provided labels.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        # --------------------------------------------------------
        # Backward pass and parameter update
        # --------------------------------------------------------

        # Reset gradients from the previous batch.
        optimizer.zero_grad()

        # Calculate gradients of the loss with respect to
        # the model parameters.
        loss.backward()

        # Prevent excessively large gradients from destabilizing
        # the training process.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update model parameters.
        optimizer.step()

        # Update the learning rate according to the scheduler.
        scheduler.step()

        # --------------------------------------------------------
        # Track training metrics
        # --------------------------------------------------------

        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)

        total_correct += (
            (predictions == labels).sum().item()
        )

        total_samples += labels.size(0)

        # Display progress every 500 batches.
        if (batch_idx + 1) % 500 == 0:

            avg_loss = total_loss / (batch_idx + 1)
            accuracy = total_correct / total_samples

            print(
                f"  Batch {batch_idx + 1}/{len(loader)} | "
                f"Loss: {avg_loss:.4f} | "
                f"Accuracy: {accuracy:.4f}"
            )

    # Calculate metrics for the complete epoch.
    epoch_loss = total_loss / len(loader)
    epoch_accuracy = total_correct / total_samples

    return epoch_loss, epoch_accuracy


def evaluate(model, loader, device):
    """
    Evaluate the model on a validation or test dataset.

    Returns:
        Loss, accuracy, precision, recall, F1-score,
        predictions, and ground-truth labels.
    """

    model.eval()

    total_loss = 0
    all_predictions = []
    all_labels = []

    # Disable gradient calculation because the model is not
    # being updated during evaluation.
    with torch.no_grad():

        for batch in loader:

            # Move the batch to the selected device.
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass.
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            # Select the class with the highest predicted probability.
            predictions = torch.argmax(logits, dim=1)

            # Store predictions and labels for calculating
            # the final evaluation metrics.
            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    # Calculate evaluation metrics.
    epoch_loss = total_loss / len(loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average='weighted',
        zero_division=0
    )

    return (
        epoch_loss,
        accuracy,
        precision,
        recall,
        f1,
        all_predictions,
        all_labels
    )

In [ ]:

# ============================================================
# MODEL TRAINING LOOP
# ============================================================

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

# Track the best validation F1-score for model selection.
best_val_f1 = 0
best_epoch = 0

# Stop training if validation F1 does not improve
# for two consecutive epochs.
patience = 2
patience_counter = 0

# Store training and validation metrics for later analysis.
training_history = {
    'epoch': [],
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_f1': []
}


for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 60)

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_loss, train_accuracy = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f}"
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_loss, val_accuracy, val_precision, val_recall, val_f1, _, _ = evaluate(
        model,
        val_loader,
        device
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )


    # --------------------------------------------------------
    # Store training history
    # --------------------------------------------------------

    training_history['epoch'].append(epoch + 1)
    training_history['train_loss'].append(train_loss)
    training_history['train_accuracy'].append(train_accuracy)
    training_history['val_loss'].append(val_loss)
    training_history['val_accuracy'].append(val_accuracy)
    training_history['val_f1'].append(val_f1)


    # --------------------------------------------------------
    # Save best model based on validation F1
    # --------------------------------------------------------

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0

        model_save_path = os.path.join(
            MODEL_OUTPUT_DIR,
            "best_model.pt"
        )

        torch.save(
            model.state_dict(),
            model_save_path
        )

        print(
            f"✓ Best model saved "
            f"(Validation F1: {val_f1:.4f})"
        )

    else:

        patience_counter += 1

        # Stop training when validation F1 has not improved
        # for the configured number of consecutive epochs.
        if patience_counter >= patience:

            print(
                f"\nEarly stopping triggered "
                f"after {epoch + 1} epochs."
            )
            break


print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

print(
    f"Best epoch: {best_epoch} "
    f"(Validation F1: {best_val_f1:.4f})"
)


# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(training_history)

history_path = os.path.join(
    RESULTS_OUTPUT_DIR,
    "training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print(f"✓ Training history saved: {history_path}")


# ============================================================
# LOAD BEST MODEL FOR FINAL EVALUATION
# ============================================================

model.load_state_dict(
    torch.load(
        os.path.join(
            MODEL_OUTPUT_DIR,
            "best_model.pt"
        )
    )
)

print("\n✓ Best model loaded for final evaluation")


STARTING TRAINING

Epoch 1/4
------------------------------------------------------------
  Batch 500/6626 | Loss: 0.5397 | Accuracy: 0.7690
  Batch 1000/6626 | Loss: 0.3089 | Accuracy: 0.8747
  Batch 1500/6626 | Loss: 0.2237 | Accuracy: 0.9124
  Batch 2000/6626 | Loss: 0.1791 | Accuracy: 0.9317
  Batch 2500/6626 | Loss: 0.1513 | Accuracy: 0.9437
  Batch 3000/6626 | Loss: 0.1312 | Accuracy: 0.9520
  Batch 3500/6626 | Loss: 0.1166 | Accuracy: 0.9579
  Batch 4000/6626 | Loss: 0.1054 | Accuracy: 0.9624
  Batch 4500/6626 | Loss: 0.0970 | Accuracy: 0.9659
  Batch 5000/6626 | Loss: 0.0897 | Accuracy: 0.9688
  Batch 5500/6626 | Loss: 0.0837 | Accuracy: 0.9712
  Batch 6000/6626 | Loss: 0.0783 | Accuracy: 0.9733


In [9]:
# ============================================================
# FINAL MODEL EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)


# ============================================================
# LOAD BEST SAVED MODEL
# ============================================================

checkpoint_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "best_model.pt"
)

print("\nLoading saved model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print(f"✓ Model loaded from: {checkpoint_path}")
print(f"Device: {device}")


# ============================================================
# EVALUATE CLEAN TEST SET
# ============================================================

print("\n" + "-" * 60)
print("CLEAN TEST SET")
print("-" * 60)

(
    clean_loss,
    clean_accuracy,
    clean_precision,
    clean_recall,
    clean_f1,
    _,
    _
) = evaluate(
    model,
    test_clean_loader,
    device
)

print(f"Test samples: {len(test_clean_dataset):,}")
print(f"Loss:         {clean_loss:.4f}")
print(f"Accuracy:     {clean_accuracy:.4f}")
print(f"Precision:    {clean_precision:.4f}")
print(f"Recall:       {clean_recall:.4f}")
print(f"F1 Score:     {clean_f1:.4f}")


# ============================================================
# EVALUATE AUGMENTED TEST SET
# ============================================================

print("\n" + "-" * 60)
print("AUGMENTED TEST SET")
print("-" * 60)

(
    augmented_loss,
    augmented_accuracy,
    augmented_precision,
    augmented_recall,
    augmented_f1,
    _,
    _
) = evaluate(
    model,
    test_augmented_loader,
    device
)

print(f"Test samples: {len(test_augmented_dataset):,}")
print(f"Loss:         {augmented_loss:.4f}")
print(f"Accuracy:     {augmented_accuracy:.4f}")
print(f"Precision:    {augmented_precision:.4f}")
print(f"Recall:       {augmented_recall:.4f}")
print(f"F1 Score:     {augmented_f1:.4f}")


# ============================================================
# ROBUSTNESS COMPARISON
# ============================================================

print("\n" + "-" * 60)
print("CLEAN VS AUGMENTED")
print("-" * 60)

f1_drop = clean_f1 - augmented_f1

print(f"Clean test F1:      {clean_f1:.4f}")
print(f"Augmented test F1:  {augmented_f1:.4f}")
print(f"F1 difference:      {f1_drop:+.4f}")


print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)


FINAL MODEL EVALUATION

Loading saved model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded from: /content/drive/MyDrive/code_switch_project/models/xlm_roberta_code_switching/best_model.pt
Device: cuda

------------------------------------------------------------
CLEAN TEST SET
------------------------------------------------------------
Test samples: 39,740
Loss:         0.1144
Accuracy:     0.9704
Precision:    0.9716
Recall:       0.9704
F1 Score:     0.9704

------------------------------------------------------------
AUGMENTED TEST SET
------------------------------------------------------------
Test samples: 39,740
Loss:         0.3819
Accuracy:     0.9254
Precision:    0.9303
Recall:       0.9254
F1 Score:     0.9251

------------------------------------------------------------
CLEAN VS AUGMENTED
------------------------------------------------------------
Clean test F1:      0.9704
Augmented test F1:  0.9251
F1 difference:      +0.0453

EVALUATION COMPLETE
